# Asteroid Collision

We are given an array `asteroids` of integers representing asteroids moving in a straight line.

- The **absolute value** is the asteroid's size.
- A **positive** number moves to the **right**; a **negative** number moves to the **left**.
- All asteroids move at the same speed.

When two asteroids collide, the smaller one explodes. If both have the same size, both explode. Asteroids moving in the same direction never collide.

**Goal:** return the state of the asteroids after all collisions.

| Input | Output | Why |
|---|---|---|
| `[5, 10, -5]` | `[5, 10]` | `10` and `-5` collide → `-5` explodes. `5` and `10` never meet. |
| `[8, -8]` | `[]` | Equal size → both explode. |
| `[10, 2, -5]` | `[10]` | `2` and `-5` collide → `2` explodes. Then `10` and `-5` collide → `-5` explodes. |
| `[-2, -1, 1, 2]` | `[-2, -1, 1, 2]` | The left-movers are already to the left of the right-movers, so nothing collides. |

## Approach: a stack

Process the asteroids left to right and keep a stack of the asteroids that have survived so far.

A collision can only happen when the **top of the stack moves right (`> 0`)** and the **incoming asteroid moves left (`< 0`)**. In every other case (same direction, or the stack top moves left and the new one moves right) the two are moving apart, so the new asteroid is simply pushed.

When a collision does happen, repeat while the collision condition still holds:

1. `abs(top) < abs(new)` → the top explodes (pop) and the new asteroid keeps going, so check the next top.
2. `abs(top) == abs(new)` → both explode (pop, and the new asteroid is gone).
3. `abs(top) > abs(new)` → the new asteroid explodes.

If the new asteroid survives every collision, push it.

In [5]:
from typing import List


def asteroid_collision(asteroids: List[int]) -> List[int]:
    stack: List[int] = []

    for ast in asteroids:
        alive = True

        # collision: stack top moves right, incoming moves left
        while alive and ast < 0 and stack and stack[-1] > 0:
            top = stack[-1]
            if top < -ast:          # top is smaller -> top explodes, keep checking
                stack.pop()
            elif top == -ast:       # same size -> both explode
                stack.pop()
                alive = False
            else:                   # top is bigger -> incoming explodes
                alive = False

        if alive:
            stack.append(ast)

    return stack

## Step-by-step trace

The helper below runs the same algorithm but prints every step, so you can watch the stack change. Try it on `[10, 2, -5]`.

In [2]:
def trace_collisions(asteroids: List[int]) -> List[int]:
    stack: List[int] = []
    print(f"input: {asteroids}\n")

    for ast in asteroids:
        print(f"-> asteroid {ast:>3} arrives   stack = {stack}")
        alive = True

        while alive and ast < 0 and stack and stack[-1] > 0:
            top = stack[-1]
            if top < -ast:
                print(f"     {top} vs {ast}: {top} explodes")
                stack.pop()
            elif top == -ast:
                print(f"     {top} vs {ast}: both explode")
                stack.pop()
                alive = False
            else:
                print(f"     {top} vs {ast}: {ast} explodes")
                alive = False

        if alive:
            stack.append(ast)
        print(f"     stack = {stack}\n")

    print(f"result: {stack}")
    return stack


trace_collisions([10, 2, -5]);

input: [10, 2, -5]

-> asteroid  10 arrives   stack = []
     stack = [10]

-> asteroid   2 arrives   stack = [10]
     stack = [10, 2]

-> asteroid  -5 arrives   stack = [10, 2]
     2 vs -5: 2 explodes
     10 vs -5: -5 explodes
     stack = [10]

result: [10]


## Tests

The examples from the task plus some edge cases: empty input, single asteroid, chain reactions, and asteroids that survive after clearing several others.

In [3]:
tests = [
    # (input, expected)
    ([5, 10, -5],            [5, 10]),
    ([8, -8],                []),
    ([10, 2, -5],            [10]),
    ([-2, -1, 1, 2],         [-2, -1, 1, 2]),
    ([],                     []),
    ([7],                    [7]),
    ([-7],                   [-7]),
    ([1, 2, 3, 4],           [1, 2, 3, 4]),          # all moving right
    ([-4, -3, -2, -1],       [-4, -3, -2, -1]),      # all moving left
    ([1, 2, 3, -10],         [-10]),                 # one left-mover wipes out everything
    ([10, 1, 2, 3, -5],      [10]),                  # chain reaction, then the big one wins
    ([3, 5, -5, -3],         []),                    # 5/-5 cancel, then 3/-3 cancel
    ([-2, 2, -2],            [-2]),                  # 2 and -2 cancel, first -2 escapes left
    ([5, -5, 5, -5],         []),                    # repeated mutual destruction
    ([2, -1, 1, -2],         []),                    # -1 dies to 2, 1 dies to -2, then 2/-2 cancel
]

for i, (given, expected) in enumerate(tests, 1):
    got = asteroid_collision(given)
    status = "PASS" if got == expected else "FAIL"
    print(f"{status}  #{i:<2} {str(given):<22} -> {str(got):<18} expected {expected}")
    assert got == expected, f"test {i} failed"

print("\nAll tests passed.")

PASS  #1  [5, 10, -5]            -> [5, 10]            expected [5, 10]
PASS  #2  [8, -8]                -> []                 expected []
PASS  #3  [10, 2, -5]            -> [10]               expected [10]
PASS  #4  [-2, -1, 1, 2]         -> [-2, -1, 1, 2]     expected [-2, -1, 1, 2]
PASS  #5  []                     -> []                 expected []
PASS  #6  [7]                    -> [7]                expected [7]
PASS  #7  [-7]                   -> [-7]               expected [-7]
PASS  #8  [1, 2, 3, 4]           -> [1, 2, 3, 4]       expected [1, 2, 3, 4]
PASS  #9  [-4, -3, -2, -1]       -> [-4, -3, -2, -1]   expected [-4, -3, -2, -1]
PASS  #10 [1, 2, 3, -10]         -> [-10]              expected [-10]
PASS  #11 [10, 1, 2, 3, -5]      -> [10]               expected [10]
PASS  #12 [3, 5, -5, -3]         -> []                 expected []
PASS  #13 [-2, 2, -2]            -> [-2]               expected [-2]
PASS  #14 [5, -5, 5, -5]         -> []                 expected []
PASS  #15

## Cross-check against a brute-force simulation

To make sure the stack solution is correct, compare it with a deliberately simple (and slow) version: repeatedly find the leftmost adjacent pair `(positive, negative)`, resolve that collision, and start over until no such pair remains. It's checked against the stack solution on thousands of random inputs.

In [4]:
import random


def brute_force(asteroids: List[int]) -> List[int]:
    arr = list(asteroids)
    while True:
        for i in range(len(arr) - 1):
            if arr[i] > 0 and arr[i + 1] < 0:      # right-mover directly left of a left-mover
                a, b = arr[i], arr[i + 1]
                if abs(a) > abs(b):
                    arr[i:i + 2] = [a]
                elif abs(a) < abs(b):
                    arr[i:i + 2] = [b]
                else:
                    arr[i:i + 2] = []
                break
        else:
            return arr


random.seed(42)
for _ in range(20_000):
    n = random.randint(0, 12)
    sample = [random.choice([-1, 1]) * random.randint(1, 8) for _ in range(n)]
    assert asteroid_collision(sample) == brute_force(sample), sample

print("20,000 random cases: stack solution matches brute force.")

20,000 random cases: stack solution matches brute force.


## Complexity

- **Time: O(n)** — each asteroid is pushed at most once and popped at most once, so the total work across the `while` loops is linear.
- **Space: O(n)** — the stack holds the survivors, which in the worst case is every asteroid (no collisions).